# Music Source Separation - Model Prototyping

This notebook helps you prototype and test the U-Net model architecture.

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from src.model import UNet, MaskBasedUNet, count_parameters
from src.loss import StemSeparationLoss, SpectralLoss
from src.utils import load_config, load_audio, stft, istft, magnitude_phase
from src.dataset import MusicSeparationDataset
from torch.utils.data import DataLoader

%matplotlib inline

## 1. Load Configuration

In [ ]:
config = load_config('../config.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("\nModel Configuration:")
for key, value in config['model'].items():
    print(f"  {key}: {value}")

## 2. Create Model Instances

In [ ]:
# Standard U-Net
unet = UNet(
    in_channels=config['model']['in_channels'],
    out_channels=config['model']['out_channels'],
    num_stems=config['model']['num_stems'],
    base_channels=config['model']['base_channels'],
    depth=config['model']['depth'],
    kernel_size=config['model']['kernel_size']
).to(device)

print(f"UNet Parameters: {count_parameters(unet):,}")

# Mask-based U-Net
mask_unet = MaskBasedUNet(
    in_channels=config['model']['in_channels'],
    out_channels=config['model']['out_channels'],
    num_stems=config['model']['num_stems'],
    base_channels=config['model']['base_channels'],
    depth=config['model']['depth'],
    kernel_size=config['model']['kernel_size']
).to(device)

print(f"MaskBasedUNet Parameters: {count_parameters(mask_unet):,}")

## 3. Test Forward Pass with Random Input

In [ ]:
# Create dummy input (batch=2, channels=2, freq=1025, time=256)
batch_size = 2
freq_bins = config['data']['n_fft'] // 2 + 1
time_frames = 256

dummy_input = torch.randn(batch_size, 2, freq_bins, time_frames).to(device)
print(f"Input shape: {dummy_input.shape}")

# Test UNet
print("\nTesting UNet...")
with torch.no_grad():
    output_unet = unet(dummy_input)
    print(f"Output shape: {output_unet.shape}")
    
    stems_unet = unet.separate_stems(dummy_input)
    print(f"Number of stems: {len(stems_unet)}")
    for i, stem in stems_unet.items():
        print(f"  Stem {i} shape: {stem.shape}")

# Test MaskBasedUNet
print("\nTesting MaskBasedUNet...")
with torch.no_grad():
    output_mask = mask_unet(dummy_input)
    print(f"Output shape: {output_mask.shape}")
    
    stems_mask = mask_unet.separate_stems(dummy_input)
    print(f"Number of stems: {len(stems_mask)}")
    for i, stem in stems_mask.items():
        print(f"  Stem {i} shape: {stem.shape}")

## 4. Test with Real Data

In [ ]:
# Load a sample from the dataset
try:
    dataset = MusicSeparationDataset(
        raw_dir=config['data']['raw_dir'],
        stems_dir=config['data']['stems_dir'],
        stems=config['data']['stems'],
        sample_rate=config['data']['sample_rate'],
        segment_length=config['data']['segment_length'],
        n_fft=config['data']['n_fft'],
        hop_length=config['data']['hop_length'],
        win_length=config['data']['win_length'],
        train=False
    )
    
    if len(dataset) > 0:
        sample = dataset[0]
        print(f"Loaded sample: {sample['song_name']}")
        print(f"Mix magnitude shape: {sample['mix_magnitude'].shape}")
        
        # Add batch dimension and move to device
        mix_mag = sample['mix_magnitude'].unsqueeze(0).to(device)
        mix_phase = sample['mix_phase'].unsqueeze(0).to(device)
        
        print(f"Input to model shape: {mix_mag.shape}")
    else:
        print("⚠️ No samples in dataset")
        mix_mag = None
        
except Exception as e:
    print(f"⚠️ Error loading dataset: {e}")
    mix_mag = None

## 5. Run Inference on Real Data

In [ ]:
if mix_mag is not None:
    print("Running inference with MaskBasedUNet...")
    mask_unet.eval()
    
    with torch.no_grad():
        # Get separated stems
        separated_stems = mask_unet.separate_stems(mix_mag)
        
        print(f"\nSeparated {len(separated_stems)} stems:")
        for i, (stem_idx, stem_mag) in enumerate(separated_stems.items()):
            stem_name = config['data']['stems'][i]
            print(f"  {stem_name}: {stem_mag.shape}")
            print(f"    Min: {stem_mag.min().item():.4f}, Max: {stem_mag.max().item():.4f}, Mean: {stem_mag.mean().item():.4f}")

## 6. Visualize Separated Spectrograms

In [ ]:
if mix_mag is not None:
    num_stems = len(separated_stems)
    fig, axes = plt.subplots(num_stems + 1, 1, figsize=(15, 4 * (num_stems + 1)))
    
    # Plot input mix
    mix_db = 20 * torch.log10(mix_mag[0, 0] + 1e-8).cpu().numpy()
    im = axes[0].imshow(mix_db, aspect='auto', origin='lower', cmap='viridis', vmin=-80, vmax=0)
    axes[0].set_title('Input Mix Spectrogram (dB)')
    axes[0].set_ylabel('Frequency Bins')
    plt.colorbar(im, ax=axes[0])
    
    # Plot separated stems
    for i, (stem_idx, stem_mag) in enumerate(separated_stems.items()):
        stem_name = config['data']['stems'][i]
        stem_db = 20 * torch.log10(stem_mag[0, 0] + 1e-8).cpu().numpy()
        
        im = axes[i+1].imshow(stem_db, aspect='auto', origin='lower', cmap='viridis', vmin=-80, vmax=0)
        axes[i+1].set_title(f'{stem_name.capitalize()} Spectrogram (dB)')
        axes[i+1].set_ylabel('Frequency Bins')
        if i == num_stems - 1:
            axes[i+1].set_xlabel('Time Frames')
        plt.colorbar(im, ax=axes[i+1])
    
    plt.tight_layout()
    plt.show()

## 7. Test Loss Function

In [ ]:
# Create loss function
criterion = StemSeparationLoss(
    stem_names=config['data']['stems'],
    spectral_weight=1.0,
    multi_stft_weight=0.0,
    spectral_loss_type='l1',
    use_multi_stft=False
)

print("Loss function created")

if mix_mag is not None:
    # Get target stems
    target_stems = {
        stem: sample['stem_magnitudes'][stem].unsqueeze(0).to(device)
        for stem in config['data']['stems']
    }
    
    # Compute loss
    pred_stems_dict = {
        config['data']['stems'][i]: stem_mag
        for i, stem_mag in separated_stems.items()
    }
    
    loss, loss_dict = criterion(pred_stems_dict, target_stems)
    
    print(f"\nTotal Loss: {loss.item():.4f}")
    print("\nLoss breakdown:")
    for key, value in loss_dict.items():
        print(f"  {key}: {value:.4f}")

## 8. Test Training Step

In [ ]:
if mix_mag is not None:
    # Create optimizer
    optimizer = torch.optim.Adam(mask_unet.parameters(), lr=1e-3)
    
    print("Testing training step...")
    
    # Training mode
    mask_unet.train()
    
    # Forward pass
    output = mask_unet(mix_mag)
    
    # Reshape output to separate stems
    batch_size, _, freq, time = output.shape
    output = output.view(batch_size, config['model']['num_stems'], config['model']['out_channels'], freq, time)
    
    pred_stems_dict = {
        stem: output[:, i, :, :, :]
        for i, stem in enumerate(config['data']['stems'])
    }
    
    # Compute loss
    loss, loss_dict = criterion(pred_stems_dict, target_stems)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    
    # Check gradients
    total_norm = 0
    for p in mask_unet.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    total_norm = total_norm ** 0.5
    
    print(f"Gradient norm: {total_norm:.4f}")
    
    # Optimizer step
    optimizer.step()
    
    print("✓ Training step completed successfully!")

## 9. Test DataLoader

In [ ]:
if len(dataset) > 0:
    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=2,
        shuffle=True,
        num_workers=0  # Use 0 for testing in notebook
    )
    
    print(f"DataLoader created with {len(dataloader)} batches")
    
    # Get one batch
    batch = next(iter(dataloader))
    
    print(f"\nBatch shapes:")
    print(f"  mix_magnitude: {batch['mix_magnitude'].shape}")
    print(f"  mix_phase: {batch['mix_phase'].shape}")
    print(f"  mix_audio: {batch['mix_audio'].shape}")
    print(f"\n  Stem magnitudes:")
    for stem, mag in batch['stem_magnitudes'].items():
        print(f"    {stem}: {mag.shape}")

## 10. Mini Training Loop (Few Iterations)

In [ ]:
if len(dataset) > 0:
    print("Running mini training loop (5 iterations)...\n")
    
    mask_unet.train()
    optimizer = torch.optim.Adam(mask_unet.parameters(), lr=1e-3)
    
    losses = []
    
    for i, batch in enumerate(dataloader):
        if i >= 5:  # Only 5 iterations
            break
        
        # Move to device
        mix_mag = batch['mix_magnitude'].to(device)
        stem_mags = {
            stem: batch['stem_magnitudes'][stem].to(device)
            for stem in config['data']['stems']
        }
        
        # Forward pass
        optimizer.zero_grad()
        output = mask_unet(mix_mag)
        
        # Reshape output
        batch_size, _, freq, time = output.shape
        output = output.view(batch_size, config['model']['num_stems'], config['model']['out_channels'], freq, time)
        
        pred_stems = {
            stem: output[:, idx, :, :, :]
            for idx, stem in enumerate(config['data']['stems'])
        }
        
        # Compute loss
        loss, loss_dict = criterion(pred_stems, stem_mags)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        print(f"Iteration {i+1}: Loss = {loss.item():.4f}")
    
    # Plot losses
    plt.figure(figsize=(10, 5))
    plt.plot(losses, marker='o')
    plt.title('Training Loss (Mini Loop)')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"\n✓ Mini training loop completed!")
    print(f"Initial loss: {losses[0]:.4f}")
    print(f"Final loss: {losses[-1]:.4f}")
    print(f"Change: {((losses[-1] - losses[0]) / losses[0] * 100):.2f}%")

## 11. Model Architecture Summary

In [ ]:
print("MaskBasedUNet Architecture:")
print(mask_unet)

print(f"\nTotal Parameters: {count_parameters(mask_unet):,}")

# Calculate model size
param_size = sum([param.nelement() * param.element_size() for param in mask_unet.parameters()])
buffer_size = sum([buf.nelement() * buf.element_size() for buf in mask_unet.buffers()])
size_mb = (param_size + buffer_size) / 1024**2

print(f"Model Size: {size_mb:.2f} MB")

## Summary

✅ Model creation and forward pass
✅ Real data inference
✅ Loss computation
✅ Training step
✅ DataLoader testing
✅ Mini training loop

**Next steps:**
1. If everything works, you're ready for full training!
2. Run: `python src/train.py --config config.yaml`
3. Monitor with TensorBoard: `tensorboard --logdir logs`